# Lekcija 20: Detekcija i podudaranje značajki

Ovdje počinje 2. dio: klasični 3D računalni vid, u kojem algoritmi sve češće rade s *više* slika prizora &mdash; kadrovima videozapisa ili snimkama više kamera &mdash; umjesto s jednom slikom. Prvi problem koji to stvara jest sljedeći: dvije slike istog prizora rijetko prikazuju isti sadržaj na istom položaju piksela, jer ga rotacija, promjena mjerila i promjena gledišta pomiču. Ova lekcija postupno uvodi podudaranje na temelju značajki, koje je otporno upravo na te promjene: najprije **detektore kutova** (Harris, Shi-Tomasi), koji pronalaze prepoznatljive točke koje se mogu ponovno detektirati, a zatim **SIFT**, koji dodaje invarijantnost na promjenu mjerila i deskriptor koji se može podudariti između dviju različitih slika istog prizora.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tenzor strukture: što čini dobru točku za podudaranje?

Za svaki piksel iz lokalnog prozora gradijenata (lekcija 12) izgradimo **tenzor strukture** dimenzija $2\times2$:

$$M = \sum_{\text{window}} \begin{bmatrix}I_x^2 & I_xI_y \\ I_xI_y & I_y^2\end{bmatrix}$$

Ovo bi trebalo izgledati poznato: to je upravo matrica kovarijance momenata iz lekcije 6, ali izgrađena iz gradijenata umjesto iz koordinata piksela. Njezine svojstvene vrijednosti $\lambda_1 \ge \lambda_2$ opisuju lokalnu strukturu intenziteta:

- obje male: **ravnomjerno** područje, gradijenti su slabi u svakom smjeru
- jedna velika, druga mala: **rub**, gradijent je jak okomito na rub, a slab duž njega
- obje velike: **kut**, gradijent je jak u svakom smjeru &mdash; upravo takva točka može se precizno ponovno locirati na drugoj slici

In [ ]:
img = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(img, (20, 20), (80, 80), 200, -1)
cv2.rectangle(img, (40, 40), (60, 60), 100, -1)
img_f = img.astype(np.float64)

Ix = cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)
Iy = cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)

window = 5
Sxx = cv2.boxFilter(Ix * Ix, -1, (window, window))
Syy = cv2.boxFilter(Iy * Iy, -1, (window, window))
Sxy = cv2.boxFilter(Ix * Iy, -1, (window, window))

plt.imshow(img, cmap='gray')
plt.title('Test image: corners, edges, and flat regions')
plt.axis('off')
plt.show()

## Harrisov detektor kutova

Izravno računanje svojstvenih vrijednosti na svakom mjestu donekle je skupo, pa su <a href="../references.html#harris-stephens-1988">Harris i Stephens (1988)</a> predložili jeftiniju zamjensku mjeru koja koristi samo $\det(M)$ i $\text{trace}(M)$ (koji se mogu izračunati bez pronalaženja svojstvenih vrijednosti):

$$R = \det(M) - k \cdot \text{trace}(M)^2, \qquad k \approx 0.04$$

$R$ je velik i pozitivan na kutovima, negativan na rubovima, a blizu nule u ravnomjernim područjima.

In [ ]:
k = 0.04
det_M = Sxx * Syy - Sxy**2
trace_M = Sxx + Syy
harris_response = det_M - k * trace_M**2

harris_cv = cv2.cornerHarris(img_f.astype(np.float32), blockSize=window, ksize=3, k=k)
print('correlation with cv2.cornerHarris:', np.corrcoef(harris_response.ravel(), harris_cv.ravel())[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Image')
im = axes[1].imshow(harris_response, cmap='coolwarm')
axes[1].set_title('Harris response R\n(red=corner, blue=edge)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Shi-Tomasi: „Dobre značajke za praćenje”

<a href="../references.html#shi-tomasi-1994">Shi i Tomasi (1994)</a> zagovarali su izravnu uporabu **najmanje svojstvene vrijednosti** matrice $M$ umjesto Harrisove zamjenske mjere s determinantom i tragom:

$$R_{\text{ST}} = \min(\lambda_1, \lambda_2)$$

Točka je izražen kut samo ako su *obje* svojstvene vrijednosti velike &mdash; upravo to znači da je njihov minimum velik. Za razliku od Harrisova $R$, ta mjera ima izravno, lako protumačivo značenje: proporcionalna je najlošijoj preciznosti praćenja u bilo kojem smjeru.

In [ ]:
discriminant = np.sqrt(np.clip((trace_M / 2)**2 - det_M, 0, None))
lambda_min = trace_M / 2 - discriminant

eigval_cv = cv2.cornerMinEigenVal(img_f.astype(np.float32), blockSize=window, ksize=3)
print('correlation with cv2.cornerMinEigenVal:', np.corrcoef(lambda_min.ravel(), eigval_cv.ravel())[0, 1])

corners = cv2.goodFeaturesToTrack(img_f.astype(np.uint8), maxCorners=20, qualityLevel=0.1, minDistance=5)

plt.imshow(img, cmap='gray')
for pt in corners[:, 0]:
    plt.scatter(*pt, c='red', s=40, marker='+')
plt.title('cv2.goodFeaturesToTrack (Shi-Tomasi)')
plt.axis('off')
plt.show()

## Problem mjerila

Harris i Shi-Tomasi rade s jednom, fiksnom veličinom prozora. Ako isti kut povećamo ili smanjimo, on pri tom fiksnom mjerilu može prestati izgledati kao kut (oštar kut postaje blaga krivulja kada ga dovoljno smanjimo) &mdash; ti detektori nisu invarijantni na promjenu mjerila. **SIFT** (Scale-Invariant Feature Transform, transformacija značajki invarijantna na mjerilo) (<a href="../references.html#lowe-2004">Lowe, 1999./2004.</a><span class="landmark-paper">&#9733;</span>) to rješava traženjem ključnih točaka u cijelom prostoru mjerila, a ne samo pri jednoj veličini prozora.

## SIFT: ekstremi u prostoru mjerila i deskriptori

Sažeti postupak SIFT-a:

1. Izgraditi **prostor mjerila razlike Gaussovih funkcija (DoG)** &mdash; upravo aproksimaciju Laplaciana pomoću DoG-a iz lekcije 13, izračunatu pri mnogim razinama zamućenja.
2. Pronaći ključne točke kao lokalne ekstreme odziva DoG-a u prostoru *i* mjerilu &mdash; točku koja je maksimum u usporedbi sa svojih 26 susjeda (uključujući 8 susjeda na trenutačnoj razini mjerila *i* po 9 susjeda na razinama iznad i ispod).
3. Svakoj ključnoj točki dodijeliti dominantnu orijentaciju iz lokalnog histograma gradijenata, kako bi deskriptori mogli biti invarijantni na rotaciju.
4. Izgraditi 128-dimenzijski deskriptor iz histograma orijentacija gradijenata u mreži $4\times4$ potpodručja oko ključne točke.

Rezultat je ključna točka s položajem, mjerilom, orijentacijom i vektorom deskriptora oblikovanim tako da bude vrlo sličan na dvjema slikama čak i uz umjerenu rotaciju, promjenu mjerila i osvjetljenja.

In [ ]:
textured = cv2.imread('../img/building.png', cv2.IMREAD_GRAYSCALE)
textured = cv2.cvtColor(textured, cv2.COLOR_BGR2RGB)

sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(textured, None)
print(f'found {len(keypoints)} keypoints, each with a {descriptors.shape[1]}-dim descriptor')

keypoint_vis = cv2.drawKeypoints(textured, keypoints, None,
                                  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(textured)
axes[0].set_title('Original')
axes[1].imshow(keypoint_vis)
axes[1].set_title('SIFT keypoints (circle size = scale, line = orientation)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Fotografija: <a href="https://unsplash.com/@dawsonlovell" target="_blank" rel="noopener">Dawson Lovell</a>, <a href="https://unsplash.com/photos/flatiron-building-in-toronto-W_MUqtuHwyY" target="_blank" rel="noopener">Unsplash</a></p>

## Podudaranje uz rotaciju i promjenu mjerila

Rotiramo i smanjujemo sliku pomoću *poznate* transformacije, zasebno detektiramo SIFT značajke na objema slikama, a zatim podudaramo deskriptore pretraživanjem najbližeg susjeda. **Loweov test omjera** zadržava podudaranje samo ako je najbolji kandidat znatno bliži od *drugog* najboljeg kandidata &mdash; jednostavan i učinkovit način odbacivanja dvosmislenih podudaranja.

In [ ]:
M = cv2.getRotationMatrix2D((150, 150), angle=35, scale=0.7)
rotated_scaled = cv2.warpAffine(textured, M, (300, 300))

kp1, des1 = sift.detectAndCompute(textured, None)
kp2, des2 = sift.detectAndCompute(rotated_scaled, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

print(f'keypoints: {len(kp1)} (original), {len(kp2)} (rotated + scaled)')
print(f'good matches after Lowe ratio test: {len(good_matches)} / {len(raw_matches)} candidate pairs')

### Koliko je podudaranja doista točno?

Budući da znamo točnu transformaciju `M` kojom je stvorena druga slika, možemo izravno provjeriti: završava li preslikavanje svake podudarene ključne točke pomoću `M` blizu mjesta s kojim je stvarno podudarena?

In [ ]:
correct = 0
for m in good_matches:
    p1 = np.array(kp1[m.queryIdx].pt + (1,))
    p2 = np.array(kp2[m.trainIdx].pt)
    predicted = M @ p1
    if np.linalg.norm(predicted - p2) < 3:
        correct += 1

print(f'geometrically correct matches: {correct} / {len(good_matches)} ({100 * correct / len(good_matches):.0f}%)')

match_vis = cv2.drawMatches(textured, kp1, rotated_scaled, kp2, good_matches, None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(11, 5))
plt.imshow(match_vis)
plt.title(f'{len(good_matches)} matches across a 35-degree rotation + 0.7x scale change')
plt.axis('off')
plt.show()

Gotovo sva podudaranja koja prođu test omjera doista su točne korespondencije, unatoč rotaciji i promjeni mjerila &mdash; nešto što detektor vezan uz jedno mjerilo i jednu orijentaciju jednostavno ne bi mogao pronaći. Upravo zato značajke poput SIFT-a čine klasičan pristup zadacima kao što su spajanje slika, prepoznavanje objekata i vizualna lokalizacija.

### Zadaci

1. Povećajte kut rotacije na 90 stupnjeva i postavite faktor mjerila na 0.3. Održavaju li se broj dobrih podudaranja i postotak geometrijski točnih podudaranja ili se smanjuju? Kada SIFT počinje imati poteškoća?
2. Isprobajte `cv2.ORB_create()` (mnogo bržu, besplatnu alternativu SIFT-u s binarnim deskriptorima) uz `cv2.NORM_HAMMING` u `BFMatcher` umjesto zadane L2 norme. Usporedite broj dobrih podudaranja i kvalitativno usporedite brzinu koristeći `%timeit` na `detectAndCompute`.
3. Snizite prag testa omjera s `0.75` na `0.6`. Kako se mijenjaju broj dobrih podudaranja i udio onih koja su geometrijski točna? Što to govori o ulozi praga u kompromisu između preciznosti i odziva?